<a href="https://colab.research.google.com/github/elijahmflomo/Sem_2_APPLIED-NATURAL-LANGUAGE-PROCESSING/blob/main/Lab_Assignment_7_ShallowParsing_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Scenario-Based 1**

A recruitment company similar to LinkedIn wants to automatically extract
useful information from candidate resumes. The system should:
Identify Noun Phrases (NPs) such as skills and job roles using Shallow
Parsing (Chunking).Extract Named Entities like Person Name,
Organization, Location using NER
Lab Task
 Develop a Python program using NLTK to:
 Perform POS tagging on resume text.
 Apply Chunking (Shallow Parsing) to extract:
 Noun Phrases (NP)Verb Phrases (VP) (optional). Perform Named
Entity Recognition (NER) to
identify:PERSON,ORGANIZATION,LOCATION,DATE

**The Workflow (The "Why" and "How")**

Tokenization: We break the raw resume text into individual words (tokens).

Part-of-Speech (POS) Tagging: We assign a grammatical category to each word (e.g., "Elon" = Noun, "Worked" = Verb). This is the foundation for everything else.

Shallow Parsing (Chunking): Instead of a full deep parse (which is slow), we look for patterns of tags. For example, an Adjective followed by a Noun usually forms a Noun Phrase (e.g., "Lead Engineer").

Named Entity Recognition (NER): We use pre-trained statistical models to identify specific categories like people, companies, and dates.

In [1]:
# Step 1: Install & Prepare NLTK

import nltk

# Download the required "brains" for NLTK
nltk.download('punkt')         # For tokenization
nltk.download('averaged_perceptron_tagger') # For POS tagging
nltk.download('maxent_ne_chunker')          # For NER
nltk.download('words')                      # For NER vocabulary

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [5]:
# Step 2: The Resume Extraction Program

from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.chunk import RegexpParser
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng') # Download specific English tagger
nltk.download('maxent_ne_chunker_tab') # Download specific NLTK NER chunker

# 1. Load a Sample Resume Text
resume_text = "Elijah M Flomo worked as a Senior Engineer at Orange GSM in Liberia since January 2024."

# 2. Perform POS Tagging
tokens = word_tokenize(resume_text)
pos_tags = pos_tag(tokens)
print("--- Step 2: POS Tags ---\n", pos_tags, "\n")

# 3. Apply Chunking (Shallow Parsing)
# We define a grammar using Regex patterns of POS tags
# NP: Optional Determiner (a/the), then any number of Adjectives, then one or more Nouns
# VP: One or more Verbs followed by an optional Preposition
grammar = r"""
    NP: {<DT>?<JJ>*<NN.?>+}
    VP: {<VB.?>+<IN>?}
"""
chunk_parser = RegexpParser(grammar)
chunked_tree = chunk_parser.parse(pos_tags)

print("--- Step 3: Extracted Phrases (Chunks) ---")
for subtree in chunked_tree.subtrees(filter=lambda t: t.label() in ['NP', 'VP']):
    phrase = " ".join([word for word, tag in subtree.leaves()])
    print(f"{subtree.label()}: {phrase}")

# 4. Named Entity Recognition (NER)
ner_tree = ne_chunk(pos_tags)

print("\n--- Step 4: Named Entities ---")
for chunk in ner_tree:
    if hasattr(chunk, 'label'):
        # Extract specific categories: PERSON, ORGANIZATION, GPE (Location), DATE
        entity_name = " ".join(c[0] for c in chunk)
        entity_type = chunk.label()
        print(f"{entity_type}: {entity_name}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.


--- Step 2: POS Tags ---
 [('Elijah', 'NNP'), ('M', 'NNP'), ('Flomo', 'NNP'), ('worked', 'VBD'), ('as', 'IN'), ('a', 'DT'), ('Senior', 'NNP'), ('Engineer', 'NNP'), ('at', 'IN'), ('Orange', 'NNP'), ('GSM', 'NNP'), ('in', 'IN'), ('Liberia', 'NNP'), ('since', 'IN'), ('January', 'NNP'), ('2024', 'CD'), ('.', '.')] 

--- Step 3: Extracted Phrases (Chunks) ---
NP: Elijah M Flomo
VP: worked as
NP: a Senior Engineer
NP: Orange GSM
NP: Liberia
NP: January

--- Step 4: Named Entities ---
PERSON: Elijah
PERSON: Flomo
ORGANIZATION: Senior Engineer
ORGANIZATION: Orange
GPE: Liberia


**Detailed Explanation of the Workflow**

Step 1: Part-of-Speech (POS) Tagging

The model looked at each word and assigned a tag based on its grammatical role.

NNP (Proper Noun): Notice that "Elijah," "Orange," and "Liberia" are all NNP. This tells the system these are specific names, not just common words.

VBD (Verb, Past Tense): "worked" is identified as the primary action.

CD (Cardinal Digit): "2024" is recognized as a number.

Step 2: Shallow Parsing (Chunking)
This is where your Regex Grammar grouped words together into meaningful "phrases."

Noun Phrases (NP): The system saw Senior (NNP) and Engineer (NNP) and combined them into one object: "a Senior Engineer". This is crucial for resumes because we want to extract the title, not just individual words.

Verb Phrases (VP): It linked the action "worked" with the preposition "as" to create the structural bridge "worked as".

Step 3: Named Entity Recognition (NER)
This is the "AI" part where the model tries to map words to real-world categories.

PERSON: It correctly identified "Elijah" and "Flomo." (In production, we would use a "joiner" to make this one entity: "Elijah Flomo").

ORGANIZATION: It correctly identified "Orange" (the GSM company) but incorrectly tagged "Senior Engineer."

GPE (Geopolitical Entity): It correctly identified "Liberia" as a location.

**Scenario-Based 2**

A tech company similar to Google is building a Bigram Language Model
for search query prediction. During testing, the model assigns zero
probability to unseen word sequences. As an NLP engineer, apply Laplace
or Good-Turing smoothing to adjust the probability estimates. Calculate
the smoothed sentence probability and compute the perplexity of a given
test sentence, then compare the results

This lab addresses one of the most famous problems in NLP: Data Sparsity. In a search engine (like Google), users often type new combinations of words. If your model hasn't seen a specific combination before, the probability becomes $0$. Since we multiply probabilities, one $0$ makes the entire sentence probability $0$, which breaks the system.We will use Laplace Smoothing (also known as Add-One Smoothing) to fix this.🏛️ The Workflow (The "Why" and "How")Count Unigrams: Count how many times each individual word appears.Count Bigrams: Count how many times pairs of words appear together.Apply Laplace Smoothing: We add $1$ to every bigram count and add the total vocabulary size $V$ to the denominator. This ensures no probability is ever truly zero.Calculate Sentence Probability: Multiply the smoothed probabilities of each bigram in the test sentence.Compute Perplexity: This is the "Score" of our model. A lower perplexity means the model is less "perplexed" (confused) by the sentence and is more confident.

**Python Implementation**

In [6]:
import math
from collections import Counter

# 1. Training Data (Our "Database" of past searches)
training_corpus = "the quick brown fox jumps over the lazy dog"
tokens = training_corpus.lower().split()

# 2. Build Unigrams and Bigrams
unigrams = Counter(tokens)
bigrams = Counter(zip(tokens, tokens[1:]))

# Vocabulary Size (V)
vocab_size = len(set(tokens))

def get_smoothed_probability(word1, word2):
    # Laplace Smoothing Formula: (Count(w1, w2) + 1) / (Count(w1) + V)
    count_bigram = bigrams.get((word1, word2), 0)
    count_unigram = unigrams.get(word1, 0)

    smoothed_prob = (count_bigram + 1) / (count_unigram + vocab_size)
    return smoothed_prob

# 3. Test Sentence (A query the model hasn't fully seen)
test_sentence = "the quick dog"
test_tokens = test_sentence.lower().split()

# 4. Calculate Sentence Probability
# P(the quick dog) = P(quick | the) * P(dog | quick)
total_prob = 1.0
print("--- Bigram Probabilities (with Laplace Smoothing) ---")

for i in range(len(test_tokens) - 1):
    w1, w2 = test_tokens[i], test_tokens[i+1]
    prob = get_smoothed_probability(w1, w2)
    total_prob *= prob
    print(f"P({w2} | {w1}) = {prob:.4f}")

# 5. Calculate Perplexity
# Perplexity = P(S) ^ (-1/N) where N is the number of words
N = len(test_tokens)
perplexity = math.pow(total_prob, -1/N)

print("\n--- Final Results ---")
print(f"Total Smoothed Sentence Probability: {total_prob:.6f}")
print(f"Model Perplexity: {perplexity:.2f}")

--- Bigram Probabilities (with Laplace Smoothing) ---
P(quick | the) = 0.2000
P(dog | quick) = 0.1111

--- Final Results ---
Total Smoothed Sentence Probability: 0.022222
Model Perplexity: 3.56


Math Breakdown

1. The Laplace Smoothing FormulaThe formula we used is:$$P(w_n | w_{n-1}) = \frac{Count(w_{n-1}, w_n) + 1}{Count(w_{n-1}) + V}$$

Where:$V$ (Vocabulary Size) = 8 (the, quick, brown, fox, jumps, over, lazy, dog).Numerator = Count of the pair + 1 (the "Add-One").Denominator = Count of the first word + Vocabulary size.


2. Calculating Individual Bigrams

$P(\text{quick} | \text{the})$:

In our training text, "the" appears 2 times.The sequence "the quick" appears 1 time.

Math: $\frac{1 + 1}{2 + 8} = \frac{2}{10} = \mathbf{0.2000}$



In our training text, "quick" appears 1 time.The sequence "quick dog" appears 0 times (unseen sequence!).

Math: $\frac{0 + 1}{1 + 8} = \frac{1}{9} \approx \mathbf{0.1111}
$



Scenario-Based 3


A digital media company similar to Netflix wants to automatically discover
hidden themes from thousands of user reviews. As a data analyst, you are
asked to implement Topic Modeling using Python (LDA) to identify
major discussion topics. Preprocess the text, build a topic model, and
extract top keywords for each topic. Evaluate the coherence of topics and
interpret how they can improve content recommendations.

The Workflow (The "Why" and "How")

1. Preprocessing: We must clean the text (lowercase, remove stop words, lemmatize) because "the" or "is" are not "topics.

2. Dictionary & Corpus Creation: LDA doesn't understand words; it understands numbers. We create a "Bag of Words" (BoW) where each word is assigned a unique ID.

3. Model Training: We tell the LDA algorithm how many topics ($K$) we think exist.

4. Evaluation: We use a Coherence Score. If the top words in a topic are "Space, Planet, Alien," they have high coherence. If they are "Space, Sandwich, Running," coherence is low.

**Python Implementation**

We need the gensim library for this. It is the industry standard for topic modeling.

In [8]:
!pip install gensim
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# 1. Sample User Reviews (Simulating Netflix data)
reviews = [
    "The action scenes were amazing and the hero was very brave",
    "A beautiful romantic story that made me cry at the end",
    "The space battle and alien planet were visually stunning",
    "I loved the chemistry between the lead actors in this comedy",
    "Fast cars and explosions make this the best action movie ever",
    "The futuristic technology and time travel plot was very complex"
]

# 2. Simple Preprocessing
# (In a real project, we would use the cleaning pipeline from Lab 1)
stop_words = set(['the', 'and', 'was', 'were', 'very', 'that', 'this', 'with', 'at'])
processed_docs = [[word for word in doc.lower().split() if word not in stop_words]
                  for doc in reviews]

# 3. Create Dictionary and Corpus (Bag of Words)
dictionary = corpora.Dictionary(processed_docs)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

# 4. Build the LDA Model
# We will look for 3 major themes
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=3, passes=10)

# 5. Extract Top Keywords for each Topic
print("--- Discovered Netflix Themes ---")
for idx, topic in lda_model.print_topics(-1):
    print(f"Theme {idx+1}: {topic}")

# 6. Evaluate Coherence
coherence_model = CoherenceModel(model=lda_model, texts=processed_docs, dictionary=dictionary, coherence='c_v')
print(f"\nTopic Coherence Score: {coherence_model.get_coherence():.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 68.0 MB/s eta 0:00:00
--- Discovered Netflix Themes ---
Theme 1: 0.069*"alien" + 0.069*"stunning" + 0.069*"planet" + 0.069*"visually" + 0.069*"battle" + 0.069*"space" + 0.017*"i" + 0.017*"comedy" + 0.017*"loved" + 0.017*"lead"
Theme 2: 0.088*"action" + 0.051*"best" + 0.051*"cars" + 0.051*"movie" + 0.051*"ever" + 0.051*"explosions" + 0.051*"make" + 0.051*"fast" + 0.050*"scenes" + 0.050*"amazing"
Theme 3: 0.038*"plot" + 0.038*"time" + 0.038*"complex" + 0.038*"travel" + 0.038*"futuristic" + 0.038*"technology" + 0.038*"end" + 0.038*"a" + 0.038*"made" + 0.038*"story"

Topic Coherence Score: 0.3101
